In [0]:
#place it in a _setup notebook inside the explorations folder

#need to run this start of each cell one time for the notebook to work, dont like having filepaths, so I did this

#import sys

#PROJECT_ROOT = "/Workspace/Users/<email>/<projectname(root)/<pyproject location> if pyproject is in root this last one isnt needed"

#if PROJECT_ROOT not in sys.path:
    #sys.path.insert(0, PROJECT_ROOT)

# EDA Silver Layer
Verifying the cleaning decisions made in clean_silver.py.

## Cleaning decisions and motivations
- **Distance units**: normalized km/k -> km, mi/miles/mile -> mi, kept h
- **Invalid units**: d (multi-day) and unknown removed per task requirements
- **Performance**: hh:mm:ss parsed to seconds for km/mi, decimal km kept for h events
- **Birth years**: filtered impossible values (< 1700 or > 2010), nulls kept
- **Average speed**: recalculated from distance and time to fix corrupt source values
  - mi events were recorded in mph, converted to km/h (x 1.60934)
- **Athlete age**: derived from year_of_event - athlete_year_of_birth
- **Null country**: 3 rows removed as country is required for athlete dimension
- **Rows removed**: 74,329 (7,117,634 -> 7,043,305)

## athlete_id investigation

### What i found
- Dataset documentation states athlete_id_raw is globally unique per athlete
- Investigation proved this is false - different athletes share the same athlete_id_raw
- Same athlete_id_raw found with different birth years in the same event
- 12,875 athlete-event combinations had duplicate athlete_id_raw values (27,791 rows)

### Approaches considered

**Option 1 - Use athlete_id_raw directly**
- Pro: simple, matches dataset documentation intent
- Pro: works correctly for identified athletes
- Con: proved to be shared between different athletes
- Con: 12,875 ambiguous athlete-event combinations remain

**Option 2 - Composite key (athlete_id_raw + birth_year + gender + country)**
- Pro: reduces duplicates by 88% (12,875 -> 1,525 ambiguous combinations)
- Pro: correctly distinguishes identified athletes with different birth years
- Con: fails for anonymous athletes (XXX country, null birth year) who have no distinguishing attributes
- Con: same anonymous athlete may get different athlete_ids across races
- Con: different anonymous athletes may share the same athlete_id

**Option 3 - Separate dim_anon_athlete table**
- Pro: cleanly separates identified and anonymous athletes
- Pro: more accurate representation of the data
- Con: only affects 1,525 combinations (0.05% of data)
- Con: adds complexity to the model, views and dashboard queries
- Con: still cannot resolve ambiguity for truly indistinguishable anonymous athletes
- Decision: not worth the complexity for 0.05% of data

**Option 4 - Use athlete_club in composite key**
- Pro: bib numbers in club column distinguish anonymous athletes within events
- Pro: increases unique combinations from 1,735,417 to 3,158,638
- Con: athlete_club changes per race so it is not a stable athlete attribute
- Con: contradicts dim_athlete design where only stable attributes belong
- Decision: excluded from composite key for this reason

### Final decision
Composite key of athlete_id_raw + year_of_birth + gender + country (Option 2):
- Reduces ambiguous cases by 88% compared to athlete_id_raw alone
- Remaining 1,525 ambiguous combinations (0.05% of data) are fundamentally unresolvable
- Anonymous athletes had their names removed for GDPR compliance
- Without original names there is no way to reliably identify them across races

rewrote this doc with help of AI, its rly very efficient at summerizing information


In [0]:
%run ../explorations/_setup

need to fix warning for dense_rank() but its included for the doc to be used, can cause degradation in too big of a dataset. will have too look into this later

In [0]:
from importlib import reload
import transformations.silver.clean_silver as silver_module
reload(silver_module)

In [0]:
from transformations.silver.clean_silver import build_silver

build_silver(spark)

In [0]:
from utils.helpers import get_table
from pyspark.sql import functions as F
df_silver = get_table("marathos.silver.obt_results")
df_silver.printSchema()

In [0]:
df_silver = get_table("marathos.silver.obt_results")

# How many athlete_id + event_id combinations have more than one result
(df_silver.groupBy("athlete_id_raw", "event_id") 
  .count() 
  .filter(F.col("count") > 1) 
  .agg(
    F.count("*").alias("ambiguous_combinations"),
    F.sum("count").alias("total_affected_rows")
  ).display())

In [0]:
(df_silver.groupBy("athlete_id", "event_id") 
  .count() 
  .filter(F.col("count") > 1) 
  .agg(
    F.count("*").alias("ambiguous_combinations"),
    F.sum("count").alias("total_affected_rows")
  ).display())

In [0]:
(df_silver.withColumn(
    "calculated_age",
    F.col("year_of_event") - F.col("athlete_year_of_birth")
).select(
    "athlete_age_category",
    "calculated_age",
    "year_of_event",
    "athlete_year_of_birth"
).filter(F.col("calculated_age").isNotNull()) 
 .limit(20).display())

In [0]:
df_silver.withColumn(
    "calculated_age",
    F.col("year_of_event") - F.col("athlete_year_of_birth")
).withColumn(
    "derived_age_category",
    F.when(F.col("calculated_age") < 23, F.concat(F.col("athlete_gender"), F.lit("U23")))
     .when(F.col("calculated_age") < 30, F.concat(F.col("athlete_gender"), F.lit("23")))
     .when(F.col("calculated_age") < 35, F.concat(F.col("athlete_gender"), F.lit("30")))
     .when(F.col("calculated_age") < 40, F.concat(F.col("athlete_gender"), F.lit("35")))
     .when(F.col("calculated_age") < 45, F.concat(F.col("athlete_gender"), F.lit("40")))
     .when(F.col("calculated_age") < 50, F.concat(F.col("athlete_gender"), F.lit("45")))
     .when(F.col("calculated_age") < 55, F.concat(F.col("athlete_gender"), F.lit("50")))
     .when(F.col("calculated_age") < 60, F.concat(F.col("athlete_gender"), F.lit("55")))
     .when(F.col("calculated_age") < 65, F.concat(F.col("athlete_gender"), F.lit("60")))
     .when(F.col("calculated_age") < 70, F.concat(F.col("athlete_gender"), F.lit("65")))
     .otherwise(F.concat(F.col("athlete_gender"), F.lit("70")))
).select(
    "athlete_age_category",
    "derived_age_category",
    "calculated_age",
    "athlete_gender"
).filter(
    F.col("athlete_age_category").isNotNull() &
    F.col("derived_age_category").isNotNull()
).limit(20).display()

In [0]:
(df_silver.withColumn(
    "calculated_hours",
    F.round(F.col("performance_seconds") / 3600, 4)
).select(
    "athlete_performance_raw",
    "performance_seconds",
    "calculated_hours"
).filter(F.col("performance_seconds").isNotNull()) 
 .limit(10).display())

In [0]:
(df_silver.groupBy("event_id") 
  .agg(
    F.count("*").alias("counted_finishers"),
    F.first("event_finishers").alias("recorded_finishers")
  ) 
  .withColumn(
    "matches",
    F.col("counted_finishers") == F.col("recorded_finishers")
  ) 
  .groupBy("matches") 
  .count() 
  .display())

In [0]:
(
    df_silver
    .filter(F.col("athlete_id").cast("long") > 999999)
    .select("athlete_id")
    .distinct()
    .count()
)

In [0]:
from pyspark.sql import functions as F
from utils.helpers import get_table
#athlete ID SHOULD be unique, but this is not the case, or the data is kinda dirty where the same athlete is listed multiple times for the same event
df_silver = get_table("marathos.silver.obt_results")

(df_silver.groupBy("athlete_id_raw", "event_id") 
  .count() 
  .filter(F.col("count") > 1) 
  .orderBy(F.desc("count")) 
  .limit(10) 
  .display())


In [0]:
from utils.helpers import get_table
df_silver = get_table("marathos.silver.obt_results")
(df_silver.select("athlete_id_raw", "athlete_id", "athlete_country", "athlete_gender") 
  .limit(10) 
  .display())

In [0]:
(df_silver.select("event_distance_raw", "event_distance_unit") 
  .filter(F.col("event_distance_raw").like("%h%")) 
  .limit(10) 
  .display())

# verify the output

In [0]:
from utils.helpers import get_table
df_silver = get_table("marathos.silver.obt_results")
print(f"Rows: {df_silver.count()}")
df_silver.printSchema()
df_silver.limit(5).display()

# check the distance units

In [0]:
from pyspark.sql import functions as F
(df_silver.groupBy("event_distance_unit") 
  .count() 
  .orderBy(F.desc("count")) 
  .display())

# verify performance_seconds looks correct

In [0]:
df_silver.select(
    "athlete_performance_raw",
    "performance_seconds"
).limit(10).display()

# check ids were created correctly

In [0]:
(df_silver.select("event_name", "event_id") 
  .distinct() 
  .orderBy("event_id") 
  .limit(10) 
  .display())

In [0]:
df_silver = get_table("marathos.silver.obt_results")
df_silver.printSchema()

In [0]:
print(f"Total rows : {df_silver.count()}")
print(f"Unique events :{df_silver.select('event_id').distinct().count()}")
print(f"Unique athletes :{df_silver.select('athlete_id').distinct().count()}")

For the DBML
these are the names on the dataset website, year of event is questionable. but it would most likely cause duplicates if placed into event and not result
 

Year of event, Event dates, Event name, Event distance/length, 
Event number of finishers, 

Athlete performance, Athlete club, 
Athlete country, Athlete year of birth, Athlete gender, 
Athlete age category, Athlete average speed, Athlete ID

 Athlete is placed into Athlete and event into event. result get the measurable outcomes

In [0]:
df_silver = get_table("marathos.silver.obt_results")

# How many athletes appear more than once in the same event?
df_silver.groupBy("athlete_id_raw", "event_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .agg(
    F.count("*").alias("duplicate_athlete_event_combinations"),
    F.sum("count").alias("total_duplicate_rows"),
    F.max("count").alias("max_appearances")
  ).display()

In [0]:
df_silver = get_table("marathos.silver.obt_results")

# Check for duplicate rows based on what should be a unique combination
(df_silver.groupBy("event_id", "athlete_id", "year_of_event") 
  .count() 
  .filter(F.col("count") > 1) 
  .orderBy(F.desc("count")) 
  .limit(10) 
  .display())

In [0]:
df_silver.filter(F.col("athlete_id") == 1013784) \
  .select("athlete_id_raw", "athlete_country", "athlete_year_of_birth", "athlete_gender", "athlete_club") \
  .distinct() \
  .display()